# Align and compare footprints with OpenET

Daily weighted and unweighted footprint products are aligned independently with OpenET, tabulated side by side, and joined to tower ET in one station loop.

Expected layout: `footprints/US-UTJ/weighted/*daily*.tif` and `footprints/US-UTJ/unweighted/*daily*.tif`. Nested folders are supported. Change `FOOTPRINT_SUBDIRS` below if your names differ. Daily filenames must contain YYYYMMDD or YYYY_MM_DD (YYYY-MM-DD also works).

“Unweighted” identifies the input footprint product; its spatial footprint values are still used as weights. It does **not** mean a uniform pixel average.

# Notebook Setup

## Dependencies

In [ ]:
%pip install -q pystac-client planetary-computer rasterio statsmodels pyarrow

In [1]:
import re
import time
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import planetary_computer as pc
import rasterio
import statsmodels.api as sm
from pystac_client import Client
from rasterio.enums import Resampling
from rasterio.errors import RasterioIOError
from rasterio.vrt import WarpedVRT


## File Locations

In [9]:
DRIVE = Path("D:/FLUX")

FFP_DIR = DRIVE / "footprints"
OPENET_DIR = DRIVE / "OpenET_Daily_Tiffs"

OUTPUT_DIR = DRIVE / "footprint_weighted_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

fluxqaqc_dir = DRIVE / "database/daily_output"

## Model Names and Station IDs

In [10]:
#station_ids = ["US-UTD","US-UTE","US-UTG","US-UTJ","US-UTL","US-UTM","US-UTP","US-UTV","US-UTW"]
station_ids = ["US-UTM",'US-CdM']
MODELS = (
    "ensemble", "disalexi", "eemetric", "geesebal",
    "ptjpl", "ssebop", "sims",
)

DATE_PATTERN = re.compile(r"(?<!\d)(\d{4}_\d{2}_\d{2})(?!\d)")
FOOTPRINT_SUBDIRS = {"weighted": "weighted", "unweighted": "unweighted"}
# Compare both tower ET variants with both footprint products.
STATION_ET_COLUMNS = ("ET", "ET_corr")
OPENET_ET_COLUMNS = ("ET_fp_weighted", "ET_fp_unweighted")
# True: use the same dates for all available combinations within each model.
# False: use every valid pair (sample sizes may differ).
COMMON_COMPARISON_DATES = True
SAVE_SCATTERS = True

## Define Custom Functions

### Alignment Functions

In [11]:
def load_footprint(path):
    with rasterio.open(path) as src:
        phi = src.read(1, masked=True).astype("float64").filled(np.nan)
        grid = {
            "crs": src.crs,
            "transform": src.transform,
            "height": src.height,
            "width": src.width,
        }

    if np.any(phi[np.isfinite(phi)] < 0):
        raise ValueError(f"Negative footprint weights in {path}")

    phi = np.where(np.isfinite(phi), phi, 0.0)

    if phi.sum() <= 0:
        raise ValueError(f"No positive footprint weights in {path}")

    return phi, grid

def load_openet_with_retry(path, grid, attempts=3):
    for attempt in range(attempts):
        try:
            return load_openet_on_grid(path, grid)
        except (RasterioIOError, OSError):
            if attempt == attempts - 1:
                raise

            delay = 5 * (2 ** attempt)  # Wait 5, then 10 seconds
            print(
                f"Read failed; retrying in {delay}s "
                f"({attempt + 1}/{attempts}): {path}"
            )
            time.sleep(delay)

def load_openet_on_grid(path, grid):
    with rasterio.open(path) as src, WarpedVRT(
        src,
        **grid,
        dtype="float64",
        nodata=np.nan,
        resampling=Resampling.bilinear,
    ) as vrt:
        et = vrt.read(1, masked=True).filled(np.nan)

    return et


def compute_footprint_weighted_et(phi, et):
    valid = (phi > 0) & np.isfinite(et)

    total_weight = phi.sum()
    valid_weight = phi[valid].sum()
    coverage = valid_weight / total_weight

    if valid_weight <= 0:
        return np.nan, np.nan, float(coverage)

    # Index valid elements directly: avoids 0 * NaN propagation.
    weights = phi[valid]
    values = et[valid]

    weighted_et = np.dot(weights, values) / valid_weight
    mean_et = values.mean()

    return float(weighted_et), float(mean_et), float(coverage)


def build_openet_index(root, station_ids):
    """Index each OpenET scene under every station in its filename."""
    index = defaultdict(list)

    # Match case-insensitively while retaining original station IDs.
    station_lookup = {
        str(station).lower(): str(station)
        for station in station_ids
    }

    for path in root.rglob("*.tif"):
        name = path.stem.lower()
        tokens = set(name.split("_"))

        date_match = DATE_PATTERN.search(name)
        if date_match is None:
            continue

        matched_models = [model for model in MODELS if model in tokens]
        if not matched_models:
            continue
        if len(matched_models) > 1:
            raise ValueError(f"Ambiguous model in filename: {path.name}")

        model = matched_models[0]
        date_text = date_match.group(1)

        matched_stations = [
            original_id
            for lower_id, original_id in station_lookup.items()
            if lower_id in tokens
        ]

        # A shared scene is available to each station it contains.
        for station_id in matched_stations:
            index[(station_id, date_text)].append((model, path))

    return index

def footprint_date_from_path(path):
    match = re.search(r"(?<!\d)(\d{4})[_-]?(\d{2})[_-]?(\d{2})(?!\d)", path.stem)
    if match is None:
        raise ValueError(f"No daily date in filename: {path.name}")
    return pd.to_datetime("".join(match.groups()), format="%Y%m%d")


def index_station_footprints(station_id, issues):
    indexed = defaultdict(dict)
    for kind, subdir in FOOTPRINT_SUBDIRS.items():
        root = FFP_DIR / station_id / subdir
        if not root.is_dir():
            issues.append({"station_id": station_id, "footprint_type": kind,
                               "path": str(root), "reason": "Missing footprint subdirectory"})
            continue
        for path in sorted(root.rglob("*")):
            if path.suffix.lower() not in (".tif", ".tiff") or "daily" not in path.stem.lower():
                continue
            try:
                date = footprint_date_from_path(path)
            except ValueError as exc:
                issues.append({"station_id": station_id, "footprint_type": kind,
                                   "path": str(path), "reason": str(exc)})
                continue
            if kind in indexed[date]:
                raise ValueError(f"Duplicate {kind} footprints for {station_id} {date.date()}: "
                                 f"{indexed[date][kind]} and {path}")
            indexed[date][kind] = path
    return indexed

### Comparison Functions

In [12]:
def combine_tower_and_openet(station_id, openet_data):
    """Join in memory; dates must be daily local calendar dates on both sides."""
    tower = pd.read_csv(fluxqaqc_dir / f"{station_id}_daily.csv")
    tower["date"] = pd.to_datetime(tower["date"], errors="raise").dt.normalize()
    if tower["date"].duplicated().any():
        raise ValueError(f"Duplicate tower dates for {station_id}; resolve before joining")
    # Station/model identifiers come from the raster table.
    tower = tower.drop(columns=["station_id", "model"], errors="ignore")
    return openet_data.merge(tower, left_on="footprint_date", right_on="date",
                             how="inner", validate="many_to_one")


def compare_station(bystation, station_id, issues):
    summaries = []
    for model, model_data in bystation.groupby("model", sort=True):
        xcols = [c for c in STATION_ET_COLUMNS if c in model_data]
        ycols = [c for c in OPENET_ET_COLUMNS
                 if c in model_data and pd.to_numeric(model_data[c], errors="coerce").notna().any()]
        for missing in set(STATION_ET_COLUMNS) - set(xcols):
            issues.append({"station_id": station_id, "model": model,
                               "reason": f"Missing tower column: {missing}"})
        model_data = model_data.copy()
        for col in xcols + ycols:
            model_data[col] = pd.to_numeric(model_data[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
        if COMMON_COMPARISON_DATES:
            model_data = model_data.dropna(subset=xcols + ycols)
        for xcol in xcols:
            for ycol in ycols:
                pairs = model_data.dropna(subset=[xcol, ycol])
                x, y = pairs[xcol], pairs[ycol]
                if len(pairs) < 3 or x.nunique() < 2:
                    issues.append({"station_id": station_id, "model": model,
                                       "station_et_col": xcol, "openet_et_col": ycol,
                                       "reason": "Need >=3 pairs and variable tower ET for OLS"})
                    continue
                # Tower ET is x; OpenET is y. Keep legitimate zero ET observations.
                design = sm.add_constant(x.rename("tower_et"), has_constant="add")
                fit = sm.OLS(y, design).fit()
                error = y.to_numpy() - x.to_numpy()
                ebr = (pd.to_numeric(pairs["ebr"], errors="coerce").replace([np.inf, -np.inf], np.nan)
                       if "ebr" in pairs else pd.Series(dtype=float))
                summaries.append({
                    "station_id": station_id, "model": model,
                    "station_et_col": xcol, "openet_et_col": ycol,
                    "footprint_type": ycol.removeprefix("ET_fp_"),
                    "N": len(pairs), "R2": fit.rsquared, "Adj_R2": fit.rsquared_adj,
                    "RMSE": float(np.sqrt(np.mean(error ** 2))),
                    "MAE": float(np.mean(np.abs(error))), "bias": float(np.mean(error)),
                    "OLS_residual_RMSE": float(np.sqrt(fit.mse_resid)),
                    "intercept": fit.params["const"], "slope": fit.params["tower_et"],
                    "avg_ebr": ebr.mean(), "med_ebr": ebr.median(),
                    "start_date": pairs["footprint_date"].min(),
                    "end_date": pairs["footprint_date"].max(),
                })
                if SAVE_SCATTERS:
                    fig, ax = plt.subplots(figsize=(7, 6))
                    ax.scatter(x, y, s=12, alpha=0.5)
                    order = np.argsort(x.to_numpy())
                    ax.plot(x.iloc[order], fit.predict(design).iloc[order], color="red", label="OLS")
                    lo = min(0.0, float(x.min()), float(y.min()))
                    hi = max(float(x.max()), float(y.max()), lo + 1)
                    ax.plot([lo, hi], [lo, hi], "k--", label="1:1")
                    ax.set(xlabel=f"Tower {xcol} (mm/day)", ylabel=f"OpenET {ycol} (mm/day)",
                           title=f"{station_id} | {model} | {xcol} vs {ycol}",
                           xlim=(lo, hi), ylim=(lo, hi))
                    ax.grid(alpha=0.3)
                    ax.legend()
                    fig.tight_layout()
                    fig.savefig(OUTPUT_DIR / f"scatter_{station_id}_{model}_{xcol}_{ycol}.png", dpi=150)
                    plt.close(fig)
    return summaries

### Filtering Functions

In [13]:
def get_landsat_data(latitude: float, longitude: float, BEG_DATE = "2020-01-01", END_DATE = "2026-09-25", verbose: bool = False):
    """Performs a search for Landsat data using the Planetary Computer STAC API and retrieves metadata for the specified location and date range.

    Args:
        latitude (float): The latitude of the location.
        longitude (float): The longitude of the location.
        BEG_DATE (str, optional): The beginning date for the search. Defaults to "2020-01-01".
        END_DATE (str, optional): The end date for the search. Defaults to "2026-09-25".
        verbose (bool, optional): If True, prints additional information. Defaults to False.

    Returns:
        _type_: _description_
    """
    cat = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1",
                    modifier=pc.sign_inplace)
    items = cat.search(collections=["landsat-c2-l2"],
                    intersects={"type": "Point", "coordinates": [longitude, latitude]},
                    datetime=f"{pd.to_datetime(BEG_DATE):%Y-%m-%d}/{pd.to_datetime(END_DATE):%Y-%m-%d}").item_collection()

    # quick table of the item-level metadata
    df = pd.DataFrame([{"id": i.id, **i.properties} for i in items])
    df["datetime"] = pd.to_datetime(df["datetime"])
    if verbose:
        pass
    else:
        df = df[['datetime',
                 'eo:cloud_cover',
                 'landsat:scene_id',
                 'platform',
                 'instruments',
                 'view:sun_azimuth',
                 'view:sun_elevation'
                 ]]
    # full MTL for one scene
    #mtl = json.load(urllib.request.urlopen(items[0].assets["mtl.json"].href))
    #attrs = mtl["LANDSAT_METADATA_FILE"]["IMAGE_ATTRIBUTES"]
    return df

# Align rasters, join tower data, and compare outputs

## Index Files

In [14]:
started = time.perf_counter()

print("Indexing OpenET files...")
openet_index = build_openet_index(OPENET_DIR, station_ids)


Indexing OpenET files...


## Align Rasters & Tabulate and Compare Results

One row per station/date/model contains `ET_fp_weighted` and `ET_fp_unweighted`, plus separate `ET_mean_*`, `unity_check_*`, `valid_weight_fraction_*`, and source paths. `ET_mean_*` is the arithmetic mean over valid positive-footprint pixels; the two footprints can cover different pixels.

Missing or invalid products remain NaN, with issues recorded in CSV. Duplicate footprint dates or OpenET scenes for the same station/date/model stop processing to avoid choosing silently. Tower dates and raster dates must represent the same local day; daily ET must be in mm/day.

Each station's table is joined to tower data and compared immediately in memory. By default, comparisons use common valid dates across the available ET columns within each model (not across models). Coverage is recorded but no minimum coverage filter is imposed. RMSE, MAE, and bias use direct OpenET-minus-tower differences; `OLS_residual_RMSE` describes residuals around the fitted line. OLS uses tower ET as x and OpenET as y, correcting the original reversed axes; slopes therefore are not directly comparable to the old output.

In [ ]:
station_oet = {}
station_comparisons = {}
skipped_files = []
summary_records = []

for station_id in station_ids:
    station_started = time.perf_counter()
    print(f"Processing {station_id}...")
    records = []
    unmatched = 0
    footprints = index_station_footprints(station_id, skipped_files)
    for date, products in sorted(footprints.items()):
        matches = openet_index.get((station_id, date.strftime("%Y_%m_%d")), ())
        if not matches:
            unmatched += 1
            continue
        by_model = defaultdict(set)
        for model, path in matches:
            by_model[model].add(path)
        for model, paths in by_model.items():
            if len(paths) != 1:
                raise ValueError(f"Duplicate OpenET scenes for {station_id}, {date.date()}, {model}: {paths}")
        loaded = {}
        for kind, path in products.items():
            try:
                loaded[kind] = load_footprint(path)
            except (ValueError, RasterioIOError, OSError) as exc:
                skipped_files.append({"station_id": station_id, "footprint_type": kind,
                                          "path": str(path), "reason": str(exc)})
        for model, paths in sorted(by_model.items()):
            openet_file = next(iter(paths))
            row = {"station_id": station_id, "model": model, "footprint_date": date,
                       "openet_file": openet_file.name, "openet_path": str(openet_file)}
            # Cache alignment only for identical grids within this scene/date.
            aligned = {}
            for kind in FOOTPRINT_SUBDIRS:
                for field in ("ET_fp", "ET_mean", "unity_check", "valid_weight_fraction"):
                    row[f"{field}_{kind}"] = np.nan
                row[f"footprint_file_{kind}"] = str(products[kind]) if kind in products else None
                if kind not in loaded:
                    continue
                phi, grid = loaded[kind]
                row[f"unity_check_{kind}"] = float(phi.sum())
                key = (str(grid["crs"]), tuple(grid["transform"]), grid["height"], grid["width"])
                try:
                    if key not in aligned:
                        aligned[key] = load_openet_with_retry(openet_file, grid)
                    fp_et, mean_et, coverage = compute_footprint_weighted_et(phi, aligned[key])
                    row.update({f"ET_fp_{kind}": fp_et, f"ET_mean_{kind}": mean_et,
                                f"valid_weight_fraction_{kind}": coverage})
                except (ValueError, RasterioIOError, OSError) as exc:
                    skipped_files.append({"station_id": station_id, "model": model,
                                              "footprint_type": kind, "path": str(openet_file), "reason": str(exc)})
            records.append(row)
    if not records:
        print(f"{station_id}: no matching daily footprint/OpenET pairs")
        continue
    df = pd.DataFrame(records).sort_values(["footprint_date", "model"]).reset_index(drop=True)
    station_oet[station_id] = df
    df.to_parquet(OUTPUT_DIR / f"station_oet_{station_id}.parquet", index=False)
    try:
        bystation = combine_tower_and_openet(station_id, df)
    except (OSError, ValueError, KeyError) as exc:
        skipped_files.append({"station_id": station_id, "reason": f"Tower join failed: {exc}"})
        print(f"{station_id}: raster table saved; tower join failed: {exc}")
        continue
    station_comparisons[station_id] = bystation
    bystation.to_parquet(OUTPUT_DIR / f"stationdata_oet_{station_id}.parquet", index=False)
    summary_records.extend(compare_station(bystation, station_id, skipped_files))
    print(f"{station_id}: {len(df):,} raster rows, {len(bystation):,} joined rows, "
          f"{unmatched} unmatched dates, {time.perf_counter() - station_started:.1f}s")

summary_columns = ["station_id", "model", "station_et_col", "openet_et_col", "footprint_type",
                   "N", "R2", "Adj_R2", "RMSE", "MAE", "bias", "OLS_residual_RMSE",
                   "intercept", "slope", "avg_ebr", "med_ebr", "start_date", "end_date"]
results = pd.DataFrame(summary_records, columns=summary_columns)
final_data = results.copy()
results.to_parquet(OUTPUT_DIR / "comparison_results.parquet", index=False)
results.to_csv(OUTPUT_DIR / "comparison_results_all_combos.csv", index=False)
# Always write the current run's issues, clearing any old log when there are none.
pd.DataFrame(skipped_files).reindex(columns=["station_id", "model", "footprint_type",
    "station_et_col", "openet_et_col", "path", "reason"]).to_csv(
        OUTPUT_DIR / "processing_issues.csv", index=False)
print(f"Completed in {time.perf_counter() - started:.1f}s; {len(skipped_files)} issues logged")

Processing US-UTM...
US-UTM: 5,888 raster rows, 5,307 joined rows, 0 unmatched dates, 15.6s
Processing US-CdM...
US-CdM: 3,183 raster rows, 3,183 joined rows, 0 unmatched dates, 9.2s
Completed in 363.5s; 1629 issues logged


In [26]:
df

,station_id,model,footprint_date,openet_file,openet_path,ET_fp_weighted,ET_mean_weighted,unity_check_weighted,valid_weight_fraction_weighted,footprint_file_weighted,ET_fp_unweighted,ET_mean_unweighted,unity_check_unweighted,valid_weight_fraction_unweighted,footprint_file_unweighted
0,US-CdM,eemetric,2023-10-25,US-CdM_eemetric_et_2023_10_25_5bb14_buffer400m...,D:\FLUX\OpenET_Daily_Tiffs\US-CdM\eemetric\US-...,NaN,NaN,NaN,NaN,D:\FLUX\footprints\US-CdM\weighted\ffp_US-CdM_...,NaN,NaN,NaN,NaN,None
1,US-CdM,ensemble,2023-10-25,US-CdM_ensemble_et_2023_10_25_45c32_buffer400m...,D:\FLUX\OpenET_Daily_Tiffs\US-CdM\ensemble\US-...,NaN,NaN,NaN,NaN,D:\FLUX\footprints\US-CdM\weighted\ffp_US-CdM_...,NaN,NaN,NaN,NaN,None
2,US-CdM,ssebop,2023-10-25,US-CdM_ssebop_et_2023_10_25_79dea_buffer400m_c...,D:\FLUX\OpenET_Daily_Tiffs\US-CdM\ssebop\US-Cd...,NaN,NaN,NaN,NaN,D:\FLUX\footprints\US-CdM\weighted\ffp_US-CdM_...,NaN,NaN,NaN,NaN,None
3,US-CdM,eemetric,2023-10-26,US-CdM_eemetric_et_2023_10_26_8059b_buffer400m...,D:\FLUX\OpenET_Daily_Tiffs\US-CdM\eemetric\US-...,NaN,NaN,NaN,NaN,D:\FLUX\footprints\US-CdM\weighted\ffp_US-CdM_...,NaN,NaN,NaN,NaN,None
4,US-CdM,ensemble,2023-10-26,US-CdM_ensemble_et_2023_10_26_8e3ca_buffer400m...,D:\FLUX\OpenET_Daily_Tiffs\US-CdM\ensemble\US-...,NaN,NaN,NaN,NaN,D:\FLUX\footprints\US-CdM\weighted\ffp_US-CdM_...,NaN,NaN,NaN,NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3178,US-CdM,ensemble,2025-10-09,US-CdM_ensemble_et_2025_10_09_9c495_buffer400m...,D:\FLUX\OpenET_Daily_Tiffs\US-CdM\ensemble\US-...,NaN,NaN,NaN,NaN,D:\FLUX\footprints\US-CdM\weighted\ffp_US-CdM_...,NaN,NaN,NaN,NaN,None
3179,US-CdM,geesebal,2025-10-09,US-CdM_geesebal_et_2025_10_09_2d34a_buffer400m...,D:\FLUX\OpenET_Daily_Tiffs\US-CdM\geesebal\US-...,NaN,NaN,NaN,NaN,D:\FLUX\footprints\US-CdM\weighted\ffp_US-CdM_...,NaN,NaN,NaN,NaN,None
3180,US-CdM,ptjpl,2025-10-09,US-CdM_ptjpl_et_2025_10_09_6e696_buffer400m_cl...,D:\FLUX\OpenET_Daily_Tiffs\US-CdM\ptjpl\US-CdM...,NaN,NaN,NaN,NaN,D:\FLUX\footprints\US-CdM\weighted\ffp_US-CdM_...,NaN,NaN,NaN,NaN,None
3181,US-CdM,sims,2025-10-09,US-CdM_sims_et_2025_10_09_1429e_buffer400m_cli...,D:\FLUX\OpenET_Daily_Tiffs\US-CdM\sims\US-CdM_...,NaN,NaN,NaN,NaN,D:\FLUX\footprints\US-CdM\weighted\ffp_US-CdM_...,NaN,NaN,NaN,NaN,None


In [16]:
results.sort_values('Adj_R2', ascending=False )

,station_id,model,station_et_col,openet_et_col,footprint_type,N,R2,Adj_R2,RMSE,MAE,bias,OLS_residual_RMSE,intercept,slope,avg_ebr,med_ebr,start_date,end_date


In [ ]:
# Optional EBR subset for any processed station (no effect on saved comparisons).
high_ebr_by_station = {
    station_id: data.loc[pd.to_numeric(data["ebr"], errors="coerce") > 0.85].copy()
    for station_id, data in station_comparisons.items() if "ebr" in data
}

In [ ]:
# Optional Landsat metadata: supply coordinates separately from station_ids.
# Example: station_coordinates = {"US-UTJ": (latitude, longitude)}
station_coordinates = {}
landsat_data = {
    station_id: get_landsat_data(latitude, longitude)
    for station_id, (latitude, longitude) in station_coordinates.items()
}
landsat_passes = pd.concat(landsat_data, names=["station_id", "row"]) if landsat_data else pd.DataFrame()